# BERT Misclassification Analysis

In [1]:
!pip install -q kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nikhileswarkomati/suicide-watch
!unzip suicide-watch.zip
import pandas as pd
df = pd.read_csv('Suicide_Detection.csv')
print(f"Dataset shape: {df.shape}")
print(df.head())

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nikhileswarkomati/suicide-watch
License(s): CC-BY-SA-4.0
Archive:  suicide-watch.zip
  inflating: Suicide_Detection.csv   
Dataset shape: (232074, 3)
   Unnamed: 0                                               text        class
0           2  Ex Wife Threatening SuicideRecently I left my ...      suicide
1           3  Am I weird I don't get affected by compliments...  non-suicide
2           4  Finally 2020 is almost over... So I can never ...  non-suicide
3           8          i need helpjust help me im crying so hard      suicide
4           9  I’m so lostHello, my name is Adam (16) and I’v...      suicide


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import random

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv("Suicide_Detection.csv").dropna()
df['label'] = df['class'].map({'non-suicide': 0, 'suicide': 1})
df_small = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=5000, random_state=42)).reset_index(drop=True)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_small['text'].tolist(), df_small['label'].tolist(), test_size=0.2, stratify=df_small['label'], random_state=42
)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_enc = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class RedditDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_ds = RedditDataset(train_enc, train_labels)
val_ds = RedditDataset(val_enc, val_labels)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader))

model.train()
for batch in tqdm(train_loader, desc="Training Epoch 1"):
    optimizer.zero_grad()
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

model.eval()
preds, true = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true.extend(batch['labels'].cpu().numpy())

false_positives = [val_texts[i] for i in range(len(true)) if preds[i] == 1 and true[i] == 0]
false_negatives = [val_texts[i] for i in range(len(true)) if preds[i] == 0 and true[i] == 1]

print("\nFalse Positives (predicted suicide, actually non-suicide):\n")
for x in random.sample(false_positives, min(5, len(false_positives))): print("-", x, "\n")
print("\nFalse Negatives (missed actual suicidal posts):\n")
for x in random.sample(false_negatives, min(5, len(false_negatives))): print("-", x, "\n")

<ipython-input-3-abcf342188f9>:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=5000, random_state=42)).reset_index(drop=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/1000 [00:00<?, ?it/s]


False Positives (predicted suicide, actually non-suicide):

- Hhhhhhhh im tired of this Trigger warning because i dont wanna hurt people or make them have a panic attack, this includes mention of self harm and other things that may possibly trigger someone



So this another self check because I haven’t done one in like a while because i was doing perfectly fine. Key word was. Lets start with the first thing, i decided to harm myself because i was stressed and didn’t talk to my parents about it. Second thing im tired of feeling like nothing. And when i try to talk to my ex about how i feel about what happened he just ignores it and talks about something else. Im tired, im so tired of pretending. Im tired of skipping lunch because i feel too sad or sick to eat. Im done im done with this stupid shit. 

- How can I fix this? I keep passing out on the bus in the morning even though when I get to the stop I am energized and am almost passing out during classes? 

- A possible suicide happe

In [4]:
print("\nClassification Report:\n")
print(classification_report(true, preds, target_names=['Non-suicide', 'Suicide']))


Classification Report:

              precision    recall  f1-score   support

 Non-suicide       0.97      0.96      0.97      1000
     Suicide       0.96      0.97      0.97      1000

    accuracy                           0.97      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       0.97      0.97      0.97      2000

